# Mansonia candidate predictions using the full STRING network

This notebook reviews the saved S2F runs for *Mansonia humeralis* and *Mansonia titillans* and compares their predictions with the candidate calls in Claudio Struchiner’s report dated 2026-08-28.

The goal is to decide whether the current predictions provide useful, traceable functional hypotheses for follow-up and whether selected predictions agree with the report. S2F scores are unitless prediction scores, not probabilities or experimental confirmation.

## Scope, inputs, and evidence limits

The prediction files and run manifests are read from the external S2F output directory. The protein FASTAs, extraction summaries, run configs, GO ontology, and report PDF are read from this workspace.

The current S2F configuration points to STRING v12.0 full links and sequences, with string_core_only = False. This widens the network context used for transfer. The 283 query proteins per species were selected upstream from reference hits in Aedes, Anopheles, and Culex; using the full STRING network does not add candidate loci excluded during that earlier selection.

The PDF reports selected gene/term examples and category summaries. Its reduced-network raw prediction matrices and its source top-10 TSV files are not available at the documented path in this workspace. This notebook therefore measures agreement with the calls printed in the PDF. It does not claim that the full STRING run is more accurate or better than the reduced-network run. The run manifests record the command and relative run-config path, but do not embed or hash the configuration contents; the fingerprints below identify the configuration files as they exist now, not provably the exact historical bytes used by the completed runs.

The report’s most-specific-term method is reproduced: retain each protein’s 100 highest scoring terms, remove terms that are ancestors of another selected term through is_a or part_of, then retain the 10 highest scoring remaining terms.

In [1]:
from __future__ import annotations

import hashlib
import json
import re
import sys
import time
from collections import defaultdict
from configparser import ConfigParser
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)

WORKSPACE_ROOT = Path("/home/marcelo_baez/paccanaro-lab/PFP-Research")
S2F_ROOT = WORKSPACE_ROOT / "S2F"
OUTPUT_ROOT = Path("/run/media/marcelo_baez/HD_Disc1/.S2F/output")
REPORT_PATH = S2F_ROOT / "S2F_candidate_gene_function_report_2026-08-28.pdf"
GO_OBO_PATH = S2F_ROOT / "go.obo"
S2F_CONFIG_PATH = S2F_ROOT / "s2f.conf"
CHUNK_ROWS = 500_000
TOP_RAW_TERMS = 100
TOP_SPECIFIC_TERMS = 10

SPECIES = {
    "mansonia_humeralis": {
        "name": "Mansonia humeralis",
        "slug": "humeralis",
        "run_config": S2F_ROOT / "conf" / "mansonia_humeralis.conf",
        "fasta": S2F_ROOT / "data/mansonia/mansonia_candidate_proteins/humeralis/humeralis_all_categories_protein.cleaned.faa",
        "extraction_summary": S2F_ROOT / "data/mansonia/mansonia_candidate_proteins/humeralis/humeralis_extraction_summary.tsv",
    },
    "mansonia_titillans": {
        "name": "Mansonia titillans",
        "slug": "titillans",
        "run_config": S2F_ROOT / "conf" / "mansonia_titillans.conf",
        "fasta": S2F_ROOT / "data/mansonia/mansonia_candidate_proteins/titillans/titillans_all_categories_protein.cleaned.faa",
        "extraction_summary": S2F_ROOT / "data/mansonia/mansonia_candidate_proteins/titillans/titillans_extraction_summary.tsv",
    },
}

CATEGORY_LABELS = {
    "1_tracheal_spiracle": "Tracheal / spiracle",
    "2_cuticle_chitin": "Cuticle / chitin",
    "3_hypoxia_signaling": "Hypoxia signaling",
    "4_chemosensory": "Chemosensory",
    "5_mechanosensory": "Mechanosensory",
}
PDF_CATEGORY_COUNTS = {
    "mansonia_humeralis": {
        "1_tracheal_spiracle": 7, "2_cuticle_chitin": 175,
        "3_hypoxia_signaling": 6, "4_chemosensory": 78,
        "5_mechanosensory": 17,
    },
    "mansonia_titillans": {
        "1_tracheal_spiracle": 9, "2_cuticle_chitin": 168,
        "3_hypoxia_signaling": 6, "4_chemosensory": 85,
        "5_mechanosensory": 15,
    },
}
PDF_PREDICTION_ROWS = {
    "mansonia_humeralis": 2_942_351,
    "mansonia_titillans": 2_980_556,
}
PREDICTION_COLUMNS = ["protein_id", "term_id", "score"]

required_paths = [
    REPORT_PATH, GO_OBO_PATH, S2F_CONFIG_PATH,
    *[spec[key] for spec in SPECIES.values() for key in ("run_config", "fasta", "extraction_summary")],
]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError("Required workspace inputs are missing:\n" + "\n".join(missing_paths))
if not OUTPUT_ROOT.is_dir():
    raise FileNotFoundError(f"External S2F output directory is unavailable: {OUTPUT_ROOT}")

go_version = next(
    line.split(":", 1)[1].strip()
    for line in GO_OBO_PATH.open()
    if line.startswith("data-version:")
)
print(f"S2F workspace: {S2F_ROOT}")
print(f"External S2F outputs: {OUTPUT_ROOT}")
print(f"Professor report: {REPORT_PATH}")
print(f"GO ontology: {GO_OBO_PATH} ({go_version})")

S2F workspace: /home/marcelo_baez/paccanaro-lab/PFP-Research/S2F
External S2F outputs: /run/media/marcelo_baez/HD_Disc1/.S2F/output
Professor report: /home/marcelo_baez/paccanaro-lab/PFP-Research/S2F/S2F_candidate_gene_function_report_2026-08-28.pdf
GO ontology: /home/marcelo_baez/paccanaro-lab/PFP-Research/S2F/go.obo (releases/2026-01-23)


In [2]:
base_config = ConfigParser(interpolation=None)
base_config.read(S2F_CONFIG_PATH)
string_links = Path(base_config.get("databases", "string_links"))
string_sequences = Path(base_config.get("databases", "string_sequences"))
string_species = Path(base_config.get("databases", "string_species"))
string_core_only = base_config.getboolean("databases", "string_core_only")
if string_core_only:
    raise ValueError("The current S2F configuration is core-only; it does not match the full-network run described here.")
if "protein.links.full.v12.0" not in string_links.name or "protein.sequences.v12.0" not in string_sequences.name:
    raise ValueError("The current configuration does not identify the expected full STRING v12.0 files.")

provenance_rows = []
RUN_MANIFESTS = {}
for alias, spec in SPECIES.items():
    output_dir = OUTPUT_ROOT / alias
    prediction_path = output_dir / "prediction.df"
    manifest_path = output_dir / "run_manifest.json"
    if not prediction_path.is_file() or not manifest_path.is_file():
        raise FileNotFoundError(f"Missing prediction.df or run_manifest.json for {alias}: {output_dir}")
    manifest = json.loads(manifest_path.read_text())
    RUN_MANIFESTS[alias] = manifest
    if manifest.get("alias") != alias:
        raise ValueError(f"Run manifest alias mismatch for {alias}: {manifest.get('alias')}")
    recorded_fasta = Path(manifest["fasta"]).resolve()
    if recorded_fasta != spec["fasta"].resolve():
        raise ValueError(f"Run manifest FASTA differs from the configured candidate FASTA for {alias}.")
    recorded_run_config = (S2F_ROOT / manifest["run_config"]).resolve()
    if recorded_run_config != spec["run_config"].resolve():
        raise ValueError(f"Run manifest config differs from the expected config for {alias}.")
    provenance_rows.append({
        "species": spec["name"],
        "alias": alias,
        "run_started_utc": manifest["started_at"],
        "command": manifest["command"],
        "interpreter": manifest["interpreter"],
        "run_config": str(spec["run_config"]),
        "run_config_sha256_current": hashlib.sha256(spec["run_config"].read_bytes()).hexdigest(),
        "candidate_fasta": str(spec["fasta"]),
        "prediction_file": str(prediction_path),
        "prediction_bytes": prediction_path.stat().st_size,
        "manifest": str(manifest_path),
    })

provenance = pd.DataFrame(provenance_rows)
display(provenance)
display(pd.DataFrame([{
    "STRING links": str(string_links),
    "STRING sequences": str(string_sequences),
    "STRING species list": str(string_species),
    "string_core_only": string_core_only,
    "GO data version": go_version,
    "base S2F config SHA-256 (current)": hashlib.sha256(S2F_CONFIG_PATH.read_bytes()).hexdigest(),
}]))

,species,alias,run_started_utc,command,interpreter,run_config,run_config_sha256_current,candidate_fasta,prediction_file,prediction_bytes,manifest
0,Mansonia humeralis,mansonia_humeralis,2026-09-21T17:07:04.407524+00:00,S2F.py predict --run-config conf/mansonia_humeralis.conf -ua,/home/marcelo_baez/anaconda3/envs/S2F/bin/python,/home/marcelo_baez/paccanaro-lab/PFP-Research/S2F/conf/mansonia_humeralis.conf,b4e943a311bba4bda6fa608718a915c8978ac13792d5e18470f0ae5c8bdb150a,/home/marcelo_baez/paccanaro-lab/PFP-Research/S2F/data/mansonia/mansonia_candidate_proteins/hume...,/run/media/marcelo_baez/HD_Disc1/.S2F/output/mansonia_humeralis/prediction.df,272951487,/run/media/marcelo_baez/HD_Disc1/.S2F/output/mansonia_humeralis/run_manifest.json
1,Mansonia titillans,mansonia_titillans,2026-09-22T12:02:10.738591+00:00,S2F.py predict --run-config conf/mansonia_titillans.conf -ua,/home/marcelo_baez/anaconda3/envs/S2F/bin/python,/home/marcelo_baez/paccanaro-lab/PFP-Research/S2F/conf/mansonia_titillans.conf,f861353642837ae4f0eeb75a1d535b6c34e34acc3653f66ba67a001afbf3fdcf,/home/marcelo_baez/paccanaro-lab/PFP-Research/S2F/data/mansonia/mansonia_candidate_proteins/titi...,/run/media/marcelo_baez/HD_Disc1/.S2F/output/mansonia_titillans/prediction.df,278816611,/run/media/marcelo_baez/HD_Disc1/.S2F/output/mansonia_titillans/run_manifest.json


,STRING links,STRING sequences,STRING species list,string_core_only,GO data version,base S2F config SHA-256 (current)
0,/run/media/marcelo_baez/HD_Disc1/.S2F/data/protein.links.full.v12.0.txt.gz,/run/media/marcelo_baez/HD_Disc1/.S2F/data/protein.sequences.v12.0.fa.gz,/run/media/marcelo_baez/HD_Disc1/.S2F/data/species.v12.0.txt,False,releases/2026-01-23,881fd85a48e32d51713a1f13baf18fa40bceb93a534ff02ffedee50a8d357520


## Candidate-set and gene-model checks

Each query protein identifier contains the candidate category and locus coordinates. The extraction summary supplies the original candidate category, reference hit, identity, protein length, and internal-stop flag. These fields are joined by the locus encoded in the FASTA identifier and compared with the run’s observed protein IDs.

In [3]:
def read_fasta_ids(path: Path) -> list[str]:
    ids = []
    with path.open() as handle:
        for line in handle:
            if line.startswith(">"):
                ids.append(line[1:].strip().split()[0])
    return ids

candidate_frames = []
candidate_category_rows = []
for alias, spec in SPECIES.items():
    fasta_ids = read_fasta_ids(spec["fasta"])
    if len(fasta_ids) != 283 or len(set(fasta_ids)) != 283:
        raise ValueError(f"{spec['name']} FASTA has {len(fasta_ids)} records and {len(set(fasta_ids))} unique IDs; expected 283.")

    summary = pd.read_csv(spec["extraction_summary"], sep="\t", dtype={"category": str, "locus": str})
    summary["locus_key"] = summary["locus"].str.replace(r"^lcl\|", "", regex=True)
    if summary["locus_key"].duplicated().any():
        raise ValueError(f"Duplicate locus keys in extraction summary for {spec['name']}.")

    frame = pd.DataFrame({"alias": alias, "protein_id": fasta_ids})
    if not frame["protein_id"].str.contains("lcl|", regex=False).all():
        raise ValueError(f"Some FASTA IDs do not contain the expected lcl| locus marker for {spec['name']}.")
    frame["locus_key"] = frame["protein_id"].str.split("lcl|", n=1, regex=False).str[1]
    frame["species"] = spec["name"]
    frame = frame.merge(summary, on="locus_key", how="left", validate="one_to_one")
    if frame["category"].isna().any():
        raise ValueError(f"Some FASTA loci do not match the extraction summary for {spec['name']}.")
    frame["category_label"] = frame["category"].map(CATEGORY_LABELS)
    if frame["category_label"].isna().any():
        raise ValueError(f"An unrecognized candidate category was found for {spec['name']}.")
    candidate_frames.append(frame)
    observed_counts = frame["category"].value_counts().to_dict()
    if observed_counts != PDF_CATEGORY_COUNTS[alias]:
        raise ValueError(f"Candidate category counts differ from the PDF for {spec['name']}: {observed_counts}")
    for category, count in observed_counts.items():
        candidate_category_rows.append({
            "species": spec["name"], "category": CATEGORY_LABELS[category],
            "candidate_proteins": int(count),
            "candidates_with_internal_stops": int(frame.loc[frame["category"].eq(category), "internal_stops"].gt(0).sum()),
        })

candidate_map = pd.concat(candidate_frames, ignore_index=True)
candidate_summary = pd.DataFrame(candidate_category_rows)
display(candidate_summary.sort_values(["species", "category"]))
print(f"Candidate protein records mapped to extraction summaries: {len(candidate_map):,} / 566")

,species,category,candidate_proteins,candidates_with_internal_stops
1,Mansonia humeralis,Chemosensory,78,7
0,Mansonia humeralis,Cuticle / chitin,175,3
4,Mansonia humeralis,Hypoxia signaling,6,1
2,Mansonia humeralis,Mechanosensory,17,1
3,Mansonia humeralis,Tracheal / spiracle,7,0
6,Mansonia titillans,Chemosensory,85,4
5,Mansonia titillans,Cuticle / chitin,168,1
9,Mansonia titillans,Hypoxia signaling,6,0
7,Mansonia titillans,Mechanosensory,15,1
8,Mansonia titillans,Tracheal / spiracle,9,0


Candidate protein records mapped to extraction summaries: 566 / 566


## GO term names and hierarchy

The notebook uses the workspace GO ontology (data version 2026-01-23). The report does not specify its GO release, so exact name matches are against the workspace ontology and may not capture version-specific term changes. The same is_a and part_of relations described in the report are read from S2F’s ontology implementation. PDF term names are matched exactly, ignoring letter case. If a label has no exact match or maps to more than one active GO term, it is reported as unresolved rather than guessed.

In [4]:
sys.path.insert(0, str(S2F_ROOT))
from GOTool.GeneOntology import GeneOntology

ontology = GeneOntology(str(GO_OBO_PATH))
ontology.build_structure()

active_terms_by_name = defaultdict(list)
term_metadata = {}
for go_id, term in ontology.terms.items():
    term_metadata[go_id] = {
        "term_name": term.name,
        "namespace": term.domain,
        "obsolete": bool(term.is_obsolete),
    }
    if not term.is_obsolete:
        active_terms_by_name[term.name.strip().casefold()].append(go_id)

def resolve_report_term(term_name: str) -> tuple[str | None, str]:
    matches = active_terms_by_name.get(term_name.strip().casefold(), [])
    if len(matches) == 1:
        return matches[0], "exact unique GO name"
    if len(matches) == 0:
        return None, "no exact GO name in workspace ontology"
    return None, "ambiguous exact GO name in workspace ontology"

def term_description(go_id: str) -> tuple[str, str, str]:
    term = term_metadata.get(str(go_id))
    if term is None:
        return str(go_id), "unknown", "GO ID absent from workspace ontology"
    status = "obsolete GO term" if term["obsolete"] else "current GO term"
    return term["term_name"], term["namespace"], status

parent_map = {
    go_id: {parent.go_id for parent in term.get_parents(relations=["is_a", "part_of"])}
    for go_id, term in ontology.terms.items()
}

def collect_ancestors(go_id: str) -> frozenset[str]:
    visited = set()
    stack = list(parent_map.get(go_id, ()))
    while stack:
        parent_id = stack.pop()
        if parent_id in visited:
            continue
        visited.add(parent_id)
        stack.extend(parent_map.get(parent_id, ()))
    return frozenset(visited)

print(f"Loaded {len(ontology.terms):,} GO terms and {sum(map(len, parent_map.values())):,} direct is_a/part_of relations.")

Loaded 48,251 GO terms and 65,919 direct is_a/part_of relations.


## Calls transcribed from the professor’s report

This reference table contains the locus/GO-term examples printed in sections 4.1–4.5 of the PDF, including the score text as reported. Rows with a shared score range preserve that range as a note; the PDF does not assign those values term by term. The weaker titillans hypoxia examples in section 4.3 are recorded separately because the PDF does not name their exact GO terms.

In [5]:
import io

REPORT_CLAIMS_TSV = """section\talias\tlocus\tterm_name\treport_score_text
4.1 Tracheal / spiracle\tmansonia_humeralis\tptg014787l_1:255566-295316(-)\tprotein phosphorylation\t0.595
4.1 Tracheal / spiracle\tmansonia_humeralis\tptg014787l_1:255566-295316(-)\tATP binding\t0.294
4.1 Tracheal / spiracle\tmansonia_humeralis\tptg014787l_1:255566-295316(-)\tprotein tyrosine kinase activity\t0.198
4.1 Tracheal / spiracle\tmansonia_humeralis\tptg001450l_1:42115-44279(+)\tprotein dimerization activity\t0.434
4.1 Tracheal / spiracle\tmansonia_humeralis\tptg005348l_1:195526-196667(-)\tprotein phosphorylation\t0.142
4.1 Tracheal / spiracle\tmansonia_humeralis\tptg005348l_1:195526-196667(-)\tATP binding\t0.116
4.1 Tracheal / spiracle\tmansonia_titillans\tptg003096l_1:370250-420919(-)\tprotein phosphorylation\t0.644
4.1 Tracheal / spiracle\tmansonia_titillans\tptg003096l_1:370250-420919(-)\tATP binding\t0.312
4.1 Tracheal / spiracle\tmansonia_titillans\tptg003096l_1:370250-420919(-)\tprotein tyrosine kinase activity\t0.210
4.1 Tracheal / spiracle\tmansonia_titillans\tptg005525l_1:19971-56472(+)\tprotein phosphorylation\t0.539
4.1 Tracheal / spiracle\tmansonia_titillans\tptg005525l_1:19971-56472(+)\tATP binding\t0.214
4.1 Tracheal / spiracle\tmansonia_titillans\tptg008361l_1:292470-294606(-)\tprotein dimerization activity\t0.434
4.2 Cuticle / chitin\tmansonia_humeralis\tptg020477l_1:129326-129676(+)\tstructural constituent of pupal chitin-based cuticle\tapproximately 1.0e-4
4.2 Cuticle / chitin\tmansonia_humeralis\tptg043735l_1:7643-8056(-)\tchitin binding\tapproximately 9.9e-5
4.2 Cuticle / chitin\tmansonia_humeralis\tptg003655l_1:27238-33715(-)\tresponse to chemical\tapproximately 9.7e-5
4.2 Cuticle / chitin\tmansonia_titillans\tptg027434l_1:45794-46150(+)\tlarval chitin-based cuticle development\tapproximately 9.9e-5
4.2 Cuticle / chitin\tmansonia_titillans\tptg006591l_1:18728-19228(-)\tlarval chitin-based cuticle development\tapproximately 9.7e-5
4.2 Cuticle / chitin\tmansonia_titillans\tptg014001l_1:182750-183645(+)\tchitin binding\tapproximately 9.7e-5
4.3 Hypoxia\tmansonia_humeralis\tptg027774l_1:110881-151369(-)\tprotein dimerization activity\t0.527
4.3 Hypoxia\tmansonia_humeralis\tptg027774l_1:110881-151369(-)\tnucleus\t0.205
4.3 Hypoxia\tmansonia_humeralis\tptg027774l_1:110881-151369(-)\ttranscription regulator complex\t0.205
4.3 Hypoxia\tmansonia_humeralis\tptg027774l_1:110881-151369(-)\tDNA-binding transcription factor activity\t0.205
4.3 Hypoxia\tmansonia_humeralis\tptg010491l_1:41029-115565(-)\tprotein dimerization activity\t0.508
4.3 Hypoxia\tmansonia_humeralis\tptg010491l_1:41029-115565(-)\tnucleus\t0.207
4.3 Hypoxia\tmansonia_humeralis\tptg010491l_1:41029-115565(-)\tDNA-binding transcription factor activity\t0.205
4.3 Hypoxia\tmansonia_humeralis\tptg031231l_1:34892-49056(-)\tnucleus\t0.209
4.3 Hypoxia\tmansonia_humeralis\tptg031231l_1:34892-49056(-)\ttranscription regulator complex\t0.209
4.3 Hypoxia\tmansonia_humeralis\tptg031231l_1:34892-49056(-)\tDNA-binding transcription factor activity\t0.209
4.3 Hypoxia\tmansonia_humeralis\tptg011496l_1:182579-212080(+)\toxidoreductase activity, acting on paired donors, with incorporation/reduction of molecular oxygen\tapproximately 0.204
4.3 Hypoxia\tmansonia_humeralis\tptg011496l_1:182579-212080(+)\tiron ion binding\t0.202
4.3 Hypoxia\tmansonia_humeralis\tptg011496l_1:182579-212080(+)\tL-ascorbic acid binding\t0.107
4.3 Hypoxia\tmansonia_titillans\tptg010067l_1:74500-144879(+)\tprotein dimerization activity\t0.536
4.3 Hypoxia\tmansonia_titillans\tptg010067l_1:74500-144879(+)\tnucleus\t0.209
4.3 Hypoxia\tmansonia_titillans\tptg010067l_1:74500-144879(+)\ttranscription regulator complex\t0.209
4.3 Hypoxia\tmansonia_titillans\tptg019752l_1:154553-189619(+)\tprotein dimerization activity\t0.521
4.3 Hypoxia\tmansonia_titillans\tptg019752l_1:154553-189619(+)\tnucleus\t0.210
4.3 Hypoxia\tmansonia_titillans\tptg019752l_1:154553-189619(+)\tDNA-binding transcription factor activity\t0.210
4.3 Hypoxia\tmansonia_titillans\tptg011330l_1:154587-189098(+)\tnucleus\t0.209
4.3 Hypoxia\tmansonia_titillans\tptg011330l_1:154587-189098(+)\ttranscription regulator complex\t0.209
4.3 Hypoxia\tmansonia_titillans\tptg011330l_1:154587-189098(+)\tDNA-binding transcription factor activity\t0.209
4.4 Chemosensory\tmansonia_humeralis\tptg020126l_1:261877-262307(-)\todorant binding\t0.324
4.4 Chemosensory\tmansonia_humeralis\tptg026683l_1:136-963(-)\tresponse to carbon dioxide\tapproximately 9.4e-5
4.4 Chemosensory\tmansonia_humeralis\tptg016989l_1:21077-34066(-)\tdendrite\trange 7.5e-5 to 8.3e-5 across listed terms
4.4 Chemosensory\tmansonia_humeralis\tptg016989l_1:21077-34066(-)\tsignal transduction\trange 7.5e-5 to 8.3e-5 across listed terms
4.4 Chemosensory\tmansonia_humeralis\tptg016989l_1:21077-34066(-)\tneuronal cell body\trange 7.5e-5 to 8.3e-5 across listed terms
4.4 Chemosensory\tmansonia_titillans\tptg001135l_1:131543-163831(+)\tregulation of multicellular organismal process\trange 9.5e-5 to 1.0e-4 across listed terms
4.4 Chemosensory\tmansonia_titillans\tptg001135l_1:131543-163831(+)\tdendrite\trange 9.5e-5 to 1.0e-4 across listed terms
4.4 Chemosensory\tmansonia_titillans\tptg004258l_1:171052-172633(-)\tprotein binding\trange 5.8e-5 to 6.0e-5 across listed terms
4.4 Chemosensory\tmansonia_titillans\tptg004258l_1:171052-172633(-)\tplasma membrane\trange 5.8e-5 to 6.0e-5 across listed terms
4.4 Chemosensory\tmansonia_titillans\tptg018317l_1:203731-217974(+)\tneuron projection\trange 5.9e-5 across listed terms
4.4 Chemosensory\tmansonia_titillans\tptg018317l_1:203731-217974(+)\tsomatodendritic compartment\trange 5.9e-5 across listed terms
4.5 Mechanosensory\tmansonia_humeralis\tptg005202l_1:143910-148202(-)\tprotein binding\t0.528
4.5 Mechanosensory\tmansonia_humeralis\tptg020373l_1:22107-81775(+)\tcalcium channel activity\t0.317
4.5 Mechanosensory\tmansonia_humeralis\tptg016376l_1:38948-119650(-)\tcalcium channel activity\t0.317
4.5 Mechanosensory\tmansonia_humeralis\tptg003952l_1:223421-238019(-)\tprotein tetramerization\t0.307
4.5 Mechanosensory\tmansonia_titillans\tptg003098l_1:149680-281827(+)\tprotein binding\t0.427
4.5 Mechanosensory\tmansonia_titillans\tptg003098l_1:149680-281827(+)\tcalcium channel activity\t0.314
4.5 Mechanosensory\tmansonia_titillans\tptg013409l_1:254107-328314(-)\tcalcium channel activity\t0.322
4.5 Mechanosensory\tmansonia_titillans\tptg010824l_1:86502-345094(-)\tprotein tetramerization\t0.309
4.5 Mechanosensory\tmansonia_titillans\tptg001371l_1:277985-281365(+)\tmonoatomic ion channel activity\t0.209
"""
report_claims = pd.read_csv(io.StringIO(REPORT_CLAIMS_TSV), sep="\t", dtype=str)
report_claims["go_id"] = None
report_claims["term_map_status"] = None
for index, row in report_claims.iterrows():
    go_id, status = resolve_report_term(row["term_name"])
    report_claims.at[index, "go_id"] = go_id
    report_claims.at[index, "term_map_status"] = status
report_claims["locus_key"] = report_claims["locus"]
report_claims = report_claims.merge(
    candidate_map[["alias", "protein_id", "locus_key", "category", "category_label"]],
    on=["alias", "locus_key"], how="left", validate="many_to_one",
)
report_claims["species"] = report_claims["alias"].map({alias: spec["name"] for alias, spec in SPECIES.items()})

QUALITATIVE_REPORT_NOTES = pd.DataFrame([
    {
        "section": "4.3 Hypoxia",
        "alias": "mansonia_titillans",
        "locus": "ptg004111l_1:42600-110999(-)",
        "report_note": "The PDF says this locus scored about 4.5e-5 with a neuronal/temperature-response profile; it does not list exact GO term names.",
    },
    {
        "section": "4.3 Hypoxia",
        "alias": "mansonia_titillans",
        "locus": "ptg018471l_1:96465-131855(+)",
        "report_note": "The PDF says this locus scored about 4.5e-5 with a neuronal/temperature-response profile; it does not list exact GO term names.",
    },
])
QUALITATIVE_REPORT_NOTES["locus_key"] = QUALITATIVE_REPORT_NOTES["locus"]
QUALITATIVE_REPORT_NOTES = QUALITATIVE_REPORT_NOTES.merge(
    candidate_map[["alias", "protein_id", "locus_key", "category_label"]],
    on=["alias", "locus_key"], how="left", validate="many_to_one",
)
print(f"Transcribed exact term calls: {len(report_claims)}")
print(f"Report loci mapped to candidate proteins: {report_claims['protein_id'].notna().sum()} / {len(report_claims)}")
display(report_claims[["section", "species", "locus", "term_name", "report_score_text", "go_id", "term_map_status"]])
display(QUALITATIVE_REPORT_NOTES)

Transcribed exact term calls: 60
Report loci mapped to candidate proteins: 60 / 60


,section,species,locus,term_name,report_score_text,go_id,term_map_status
0,4.1 Tracheal / spiracle,Mansonia humeralis,ptg014787l_1:255566-295316(-),protein phosphorylation,0.595,GO:0006468,exact unique GO name
1,4.1 Tracheal / spiracle,Mansonia humeralis,ptg014787l_1:255566-295316(-),ATP binding,0.294,GO:0005524,exact unique GO name
2,4.1 Tracheal / spiracle,Mansonia humeralis,ptg014787l_1:255566-295316(-),protein tyrosine kinase activity,0.198,GO:0004713,exact unique GO name
3,4.1 Tracheal / spiracle,Mansonia humeralis,ptg001450l_1:42115-44279(+),protein dimerization activity,0.434,GO:0046983,exact unique GO name
4,4.1 Tracheal / spiracle,Mansonia humeralis,ptg005348l_1:195526-196667(-),protein phosphorylation,0.142,GO:0006468,exact unique GO name
5,4.1 Tracheal / spiracle,Mansonia humeralis,ptg005348l_1:195526-196667(-),ATP binding,0.116,GO:0005524,exact unique GO name
6,4.1 Tracheal / spiracle,Mansonia titillans,ptg003096l_1:370250-420919(-),protein phosphorylation,0.644,GO:0006468,exact unique GO name
7,4.1 Tracheal / spiracle,Mansonia titillans,ptg003096l_1:370250-420919(-),ATP binding,0.312,GO:0005524,exact unique GO name
8,4.1 Tracheal / spiracle,Mansonia titillans,ptg003096l_1:370250-420919(-),protein tyrosine kinase activity,0.210,GO:0004713,exact unique GO name
9,4.1 Tracheal / spiracle,Mansonia titillans,ptg005525l_1:19971-56472(+),protein phosphorylation,0.539,GO:0006468,exact unique GO name


,section,alias,locus,report_note,locus_key,protein_id,category_label
0,4.3 Hypoxia,mansonia_titillans,ptg004111l_1:42600-110999(-),The PDF says this locus scored about 4.5e-5 with a neuronal/temperature-response profile; it doe...,ptg004111l_1:42600-110999(-),titillans_3_hypoxia_signaling_lcl|ptg004111l_1:42600-110999(-),Hypoxia signaling
1,4.3 Hypoxia,mansonia_titillans,ptg018471l_1:96465-131855(+),The PDF says this locus scored about 4.5e-5 with a neuronal/temperature-response profile; it doe...,ptg018471l_1:96465-131855(+),titillans_3_hypoxia_signaling_lcl|ptg018471l_1:96465-131855(+),Hypoxia signaling


## Scan the raw full-network predictions

Prediction matrices are read in chunks. The notebook retains only the top 100 raw terms per protein for the report’s specificity filter, plus every nonzero score for PDF-highlighted candidate proteins so their exact GO-term ranks can be checked even when a call falls below the top 100. It verifies that the prediction matrix contains exactly the 283 query IDs in each FASTA.

In [6]:
def scan_prediction_file(alias: str, report_proteins: set[str]) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    prediction_path = OUTPUT_ROOT / alias / "prediction.df"
    best_by_protein: dict[str, pd.DataFrame] = {}
    report_chunks = []
    observed_proteins = set()
    row_count = 0
    start = time.perf_counter()

    for chunk_number, chunk in enumerate(pd.read_csv(
        prediction_path,
        sep="\t",
        header=None,
        names=PREDICTION_COLUMNS,
        usecols=[0, 1, 2],
        dtype={"protein_id": str, "term_id": str, "score": np.float64},
        na_filter=False,
        chunksize=CHUNK_ROWS,
    ), start=1):
        if chunk[["protein_id", "term_id", "score"]].isna().any().any():
            raise ValueError(f"Missing prediction values in {prediction_path}")
        if not np.isfinite(chunk["score"].to_numpy()).all():
            raise ValueError(f"Non-finite scores in {prediction_path}")
        row_count += len(chunk)
        observed_proteins.update(chunk["protein_id"].unique())
        target_chunk = chunk[chunk["protein_id"].isin(report_proteins)]
        if not target_chunk.empty:
            report_chunks.append(target_chunk.copy())

        for protein_id, group in chunk.groupby("protein_id", sort=False):
            old = best_by_protein.get(protein_id)
            combined = group if old is None else pd.concat([old, group], ignore_index=True)
            best_by_protein[protein_id] = (
                combined.sort_values(["score", "term_id"], ascending=[False, True], kind="mergesort")
                .head(TOP_RAW_TERMS)
                .reset_index(drop=True)
            )
        print(f"{alias}: read {row_count:,} prediction rows in {chunk_number} chunks")

    candidate_ids = set(candidate_map.loc[candidate_map["alias"].eq(alias), "protein_id"])
    missing = sorted(candidate_ids - observed_proteins)
    unexpected = sorted(observed_proteins - candidate_ids)
    if missing or unexpected:
        raise ValueError(
            f"Prediction protein IDs differ from the candidate FASTA for {alias}. "
            f"Missing={missing[:5]}, unexpected={unexpected[:5]}"
        )
    if len(best_by_protein) != 283:
        raise ValueError(f"{alias} has {len(best_by_protein)} prediction proteins; expected 283.")

    top100 = pd.concat(best_by_protein.values(), ignore_index=True)
    report_rows = pd.concat(report_chunks, ignore_index=True) if report_chunks else pd.DataFrame(columns=PREDICTION_COLUMNS)
    duplicate_pairs = int(report_rows.duplicated(["protein_id", "term_id"]).sum())
    if duplicate_pairs:
        raise ValueError(f"{alias} contains {duplicate_pairs} duplicate protein/GO rows among report candidate loci.")

    stats = {
        "alias": alias,
        "prediction_rows": row_count,
        "prediction_proteins": len(observed_proteins),
        "report_candidate_proteins_retained": len(report_proteins & observed_proteins),
        "file_bytes": prediction_path.stat().st_size,
        "scan_seconds": round(time.perf_counter() - start, 2),
    }
    return top100, report_rows, stats

report_proteins_by_alias = {
    alias: set(report_claims.loc[
        report_claims["alias"].eq(alias) & report_claims["protein_id"].notna(), "protein_id"
    ])
    for alias in SPECIES
}
top100_by_alias = {}
raw_report_by_alias = {}
prediction_stats = []
for alias in SPECIES:
    top100_by_alias[alias], raw_report_by_alias[alias], stats = scan_prediction_file(
        alias, report_proteins_by_alias[alias]
    )
    prediction_stats.append(stats)

prediction_qc = pd.DataFrame(prediction_stats)
prediction_qc["PDF_matrix_rows"] = prediction_qc["alias"].map(PDF_PREDICTION_ROWS)
prediction_qc["row_count_difference"] = prediction_qc["prediction_rows"] - prediction_qc["PDF_matrix_rows"]
prediction_qc["species"] = prediction_qc["alias"].map({alias: spec["name"] for alias, spec in SPECIES.items()})
display(prediction_qc[[
    "species", "prediction_rows", "prediction_proteins", "report_candidate_proteins_retained",
    "file_bytes", "PDF_matrix_rows", "row_count_difference", "scan_seconds",
]])

mansonia_humeralis: read 500,000 prediction rows in 1 chunks
mansonia_humeralis: read 1,000,000 prediction rows in 2 chunks
mansonia_humeralis: read 1,500,000 prediction rows in 3 chunks
mansonia_humeralis: read 2,000,000 prediction rows in 4 chunks
mansonia_humeralis: read 2,500,000 prediction rows in 5 chunks
mansonia_humeralis: read 2,964,991 prediction rows in 6 chunks
mansonia_titillans: read 500,000 prediction rows in 1 chunks
mansonia_titillans: read 1,000,000 prediction rows in 2 chunks
mansonia_titillans: read 1,500,000 prediction rows in 3 chunks
mansonia_titillans: read 2,000,000 prediction rows in 4 chunks
mansonia_titillans: read 2,500,000 prediction rows in 5 chunks
mansonia_titillans: read 3,000,000 prediction rows in 6 chunks
mansonia_titillans: read 3,003,196 prediction rows in 7 chunks


,species,prediction_rows,prediction_proteins,report_candidate_proteins_retained,file_bytes,PDF_matrix_rows,row_count_difference,scan_seconds
0,Mansonia humeralis,2964991,283,17,272951487,2942351,22640,4.42
1,Mansonia titillans,3003196,283,16,278816611,2980556,22640,4.42


## Most-specific terms and report-call matching

A PDF call is an exact match only when the same candidate locus has the same GO concept in its current top 10 most-specific terms. Calls found in the raw matrix but outside that list remain visible as lower-ranked predictions. A term absent from the nonzero matrix is reported as not found, not treated as experimental evidence against the function.

In [7]:
def top_specific_terms(top100: pd.DataFrame) -> pd.DataFrame:
    selected = []
    for protein_id, group in top100.groupby("protein_id", sort=False):
        candidate_go_ids = set(group["term_id"].astype(str))
        ancestor_ids = set()
        for go_id in candidate_go_ids:
            term = ontology.terms.get(go_id)
            if term is not None and not term.is_obsolete:
                ancestor_ids.update(collect_ancestors(go_id))
        specific = group.loc[~group["term_id"].astype(str).isin(ancestor_ids)].copy()
        specific = specific.sort_values(
            ["score", "term_id"], ascending=[False, True], kind="mergesort"
        ).head(TOP_SPECIFIC_TERMS)
        specific["specific_rank"] = np.arange(1, len(specific) + 1)
        selected.append(specific)

    result = pd.concat(selected, ignore_index=True)
    descriptions = result["term_id"].astype(str).map(term_description)
    result[["term_name", "namespace", "ontology_status"]] = pd.DataFrame(
        descriptions.tolist(), index=result.index
    )
    return result

specific_frames = []
raw_report_frames = []
for alias in SPECIES:
    current = top_specific_terms(top100_by_alias[alias])
    current["alias"] = alias
    specific_frames.append(current)

    raw_report = raw_report_by_alias[alias].copy()
    if not raw_report.empty:
        raw_report["raw_rank"] = raw_report.groupby("protein_id")["score"].rank(
            method="min", ascending=False
        )
        raw_report["alias"] = alias
        raw_report_frames.append(raw_report)

top10_specific = pd.concat(specific_frames, ignore_index=True)
candidate_columns = [
    "alias", "species", "protein_id", "locus_key", "category", "category_label",
    "best_hit_species", "best_hit_qid", "identity", "protein_len", "internal_stops",
]
top10_specific = top10_specific.merge(
    candidate_map[candidate_columns], on=["alias", "protein_id"], how="left", validate="many_to_one"
)
raw_report_predictions = pd.concat(raw_report_frames, ignore_index=True)
if top10_specific["category"].isna().any():
    raise ValueError("Some current top-specific predictions could not be assigned to a candidate category.")
if top10_specific.groupby("alias")["protein_id"].nunique().to_dict() != {
    "mansonia_humeralis": 283, "mansonia_titillans": 283
}:
    raise ValueError("Most-specific-term output does not cover all 283 proteins in each species.")

top10_specific["score"] = top10_specific["score"].astype(float)
print(f"Top-10 most-specific predictions retained: {len(top10_specific):,} rows")
display(top10_specific.head(12))

Top-10 most-specific predictions retained: 5,660 rows


,protein_id,term_id,score,specific_rank,term_name,namespace,ontology_status,alias,species,locus_key,category,category_label,best_hit_species,best_hit_qid,identity,protein_len,internal_stops
0,humeralis_1_tracheal_spiracle_lcl|ptg001450l_1:42115-44279(+),GO:0046983,0.591304,1,protein dimerization activity,molecular_function,current GO term,mansonia_humeralis,Mansonia humeralis,ptg001450l_1:42115-44279(+),1_tracheal_spiracle,Tracheal / spiracle,Aedes_aegypti,XP_021694596.1,0.8647,601,0
1,humeralis_1_tracheal_spiracle_lcl|ptg001450l_1:42115-44279(+),GO:0000977,0.192242,2,RNA polymerase II transcription regulatory region sequence-specific DNA binding,molecular_function,current GO term,mansonia_humeralis,Mansonia humeralis,ptg001450l_1:42115-44279(+),1_tracheal_spiracle,Tracheal / spiracle,Aedes_aegypti,XP_021694596.1,0.8647,601,0
2,humeralis_1_tracheal_spiracle_lcl|ptg001450l_1:42115-44279(+),GO:0000981,0.192242,3,"DNA-binding transcription factor activity, RNA polymerase II-specific",molecular_function,current GO term,mansonia_humeralis,Mansonia humeralis,ptg001450l_1:42115-44279(+),1_tracheal_spiracle,Tracheal / spiracle,Aedes_aegypti,XP_021694596.1,0.8647,601,0
3,humeralis_1_tracheal_spiracle_lcl|ptg001450l_1:42115-44279(+),GO:0005634,0.106449,4,nucleus,cellular_component,current GO term,mansonia_humeralis,Mansonia humeralis,ptg001450l_1:42115-44279(+),1_tracheal_spiracle,Tracheal / spiracle,Aedes_aegypti,XP_021694596.1,0.8647,601,0
4,humeralis_1_tracheal_spiracle_lcl|ptg001450l_1:42115-44279(+),GO:0005667,0.106342,5,transcription regulator complex,cellular_component,current GO term,mansonia_humeralis,Mansonia humeralis,ptg001450l_1:42115-44279(+),1_tracheal_spiracle,Tracheal / spiracle,Aedes_aegypti,XP_021694596.1,0.8647,601,0
5,humeralis_1_tracheal_spiracle_lcl|ptg001450l_1:42115-44279(+),GO:0046914,0.099734,6,transition metal ion binding,molecular_function,current GO term,mansonia_humeralis,Mansonia humeralis,ptg001450l_1:42115-44279(+),1_tracheal_spiracle,Tracheal / spiracle,Aedes_aegypti,XP_021694596.1,0.8647,601,0
6,humeralis_1_tracheal_spiracle_lcl|ptg001450l_1:42115-44279(+),GO:0001666,0.099685,7,response to hypoxia,biological_process,current GO term,mansonia_humeralis,Mansonia humeralis,ptg001450l_1:42115-44279(+),1_tracheal_spiracle,Tracheal / spiracle,Aedes_aegypti,XP_021694596.1,0.8647,601,0
7,humeralis_1_tracheal_spiracle_lcl|ptg001450l_1:42115-44279(+),GO:0140096,0.099672,8,"catalytic activity, acting on a protein",molecular_function,current GO term,mansonia_humeralis,Mansonia humeralis,ptg001450l_1:42115-44279(+),1_tracheal_spiracle,Tracheal / spiracle,Aedes_aegypti,XP_021694596.1,0.8647,601,0
8,humeralis_1_tracheal_spiracle_lcl|ptg001450l_1:42115-44279(+),GO:0071944,0.099649,9,cell periphery,cellular_component,current GO term,mansonia_humeralis,Mansonia humeralis,ptg001450l_1:42115-44279(+),1_tracheal_spiracle,Tracheal / spiracle,Aedes_aegypti,XP_021694596.1,0.8647,601,0
9,humeralis_1_tracheal_spiracle_lcl|ptg001450l_1:42115-44279(+),GO:0003008,0.099647,10,system process,biological_process,current GO term,mansonia_humeralis,Mansonia humeralis,ptg001450l_1:42115-44279(+),1_tracheal_spiracle,Tracheal / spiracle,Aedes_aegypti,XP_021694596.1,0.8647,601,0


In [8]:
comparison = report_claims.merge(
    raw_report_predictions[["alias", "protein_id", "term_id", "score", "raw_rank"]].rename(
        columns={"term_id": "go_id", "score": "current_score"}
    ),
    on=["alias", "protein_id", "go_id"], how="left", validate="many_to_one",
)
comparison = comparison.merge(
    top10_specific[["alias", "protein_id", "term_id", "score", "specific_rank"]].rename(
        columns={"term_id": "go_id", "score": "current_specific_score"}
    ),
    on=["alias", "protein_id", "go_id"], how="left", validate="many_to_one",
)

def classify_call(row) -> str:
    if pd.isna(row["protein_id"]):
        return "PDF locus not in current candidate set"
    if row["term_map_status"] != "exact unique GO name":
        return "PDF GO name unresolved"
    if pd.isna(row["current_score"]):
        return "term not found in nonzero prediction matrix"
    if pd.notna(row["specific_rank"]):
        return "exact locus and GO term in current top 10"
    return "term present in matrix, outside current top 10"

comparison["alignment"] = comparison.apply(classify_call, axis=1)
comparison = comparison.sort_values(["section", "alias", "locus", "specific_rank", "term_name"], na_position="last")
comparison_summary = (
    comparison.groupby(["section", "alias", "alignment"], dropna=False)
    .size().rename("reported_term_calls").reset_index()
)
display(comparison[[
    "section", "species", "category_label", "locus", "term_name", "go_id",
    "report_score_text", "current_score", "raw_rank", "specific_rank",
    "term_map_status", "alignment",
]])
display(comparison_summary)

,section,species,category_label,locus,term_name,go_id,report_score_text,current_score,raw_rank,specific_rank,term_map_status,alignment
3,4.1 Tracheal / spiracle,Mansonia humeralis,Tracheal / spiracle,ptg001450l_1:42115-44279(+),protein dimerization activity,GO:0046983,0.434,0.591304,4.0,1.0,exact unique GO name,exact locus and GO term in current top 10
4,4.1 Tracheal / spiracle,Mansonia humeralis,Tracheal / spiracle,ptg005348l_1:195526-196667(-),protein phosphorylation,GO:0006468,0.142,0.137125,15.0,1.0,exact unique GO name,exact locus and GO term in current top 10
5,4.1 Tracheal / spiracle,Mansonia humeralis,Tracheal / spiracle,ptg005348l_1:195526-196667(-),ATP binding,GO:0005524,0.116,0.114129,30.0,2.0,exact unique GO name,exact locus and GO term in current top 10
0,4.1 Tracheal / spiracle,Mansonia humeralis,Tracheal / spiracle,ptg014787l_1:255566-295316(-),protein phosphorylation,GO:0006468,0.595,0.539620,15.0,1.0,exact unique GO name,exact locus and GO term in current top 10
1,4.1 Tracheal / spiracle,Mansonia humeralis,Tracheal / spiracle,ptg014787l_1:255566-295316(-),ATP binding,GO:0005524,0.294,0.272042,30.0,2.0,exact unique GO name,exact locus and GO term in current top 10
2,4.1 Tracheal / spiracle,Mansonia humeralis,Tracheal / spiracle,ptg014787l_1:255566-295316(-),protein tyrosine kinase activity,GO:0004713,0.198,0.275110,26.0,NaN,exact unique GO name,"term present in matrix, outside current top 10"
6,4.1 Tracheal / spiracle,Mansonia titillans,Tracheal / spiracle,ptg003096l_1:370250-420919(-),protein phosphorylation,GO:0006468,0.644,0.582251,18.0,1.0,exact unique GO name,exact locus and GO term in current top 10
7,4.1 Tracheal / spiracle,Mansonia titillans,Tracheal / spiracle,ptg003096l_1:370250-420919(-),ATP binding,GO:0005524,0.312,0.287527,30.0,2.0,exact unique GO name,exact locus and GO term in current top 10
8,4.1 Tracheal / spiracle,Mansonia titillans,Tracheal / spiracle,ptg003096l_1:370250-420919(-),protein tyrosine kinase activity,GO:0004713,0.210,0.295377,21.0,NaN,exact unique GO name,"term present in matrix, outside current top 10"
9,4.1 Tracheal / spiracle,Mansonia titillans,Tracheal / spiracle,ptg005525l_1:19971-56472(+),protein phosphorylation,GO:0006468,0.539,0.491825,18.0,1.0,exact unique GO name,exact locus and GO term in current top 10


,section,alias,alignment,reported_term_calls
0,4.1 Tracheal / spiracle,mansonia_humeralis,exact locus and GO term in current top 10,5
1,4.1 Tracheal / spiracle,mansonia_humeralis,"term present in matrix, outside current top 10",1
2,4.1 Tracheal / spiracle,mansonia_titillans,exact locus and GO term in current top 10,5
3,4.1 Tracheal / spiracle,mansonia_titillans,"term present in matrix, outside current top 10",1
4,4.2 Cuticle / chitin,mansonia_humeralis,exact locus and GO term in current top 10,1
5,4.2 Cuticle / chitin,mansonia_humeralis,"term present in matrix, outside current top 10",2
6,4.2 Cuticle / chitin,mansonia_titillans,exact locus and GO term in current top 10,3
7,4.3 Hypoxia,mansonia_humeralis,PDF GO name unresolved,1
8,4.3 Hypoxia,mansonia_humeralis,exact locus and GO term in current top 10,7
9,4.3 Hypoxia,mansonia_humeralis,"term present in matrix, outside current top 10",5


## Candidate-wide category patterns

The next tables summarize which GO terms occur most often in the top-10 most-specific predictions within each candidate category. Counts are proteins carrying a term, not score averages; this avoids comparing S2F score magnitudes across different proteins or categories. The shared-term table requires the same GO ID to occur in both species within the same candidate category. A coverage table also reports how many candidates in each category received any and all 10 most-specific terms.

In [9]:
category_term_coverage = (
    top10_specific.groupby(
        ["alias", "species", "category", "category_label", "term_id", "term_name", "namespace"],
        as_index=False,
    )
    .agg(protein_count=("protein_id", "nunique"))
)
candidate_totals = candidate_map.groupby(["alias", "category"]).size().rename("candidate_count").reset_index()
category_term_coverage = category_term_coverage.merge(
    candidate_totals, on=["alias", "category"], how="left", validate="many_to_one"
)
category_term_coverage["fraction_of_category"] = (
    category_term_coverage["protein_count"] / category_term_coverage["candidate_count"]
)
category_term_coverage = category_term_coverage.sort_values(
    ["alias", "category", "protein_count", "term_name"],
    ascending=[True, True, False, True],
)
common_terms = category_term_coverage.pivot_table(
    index=["category", "category_label", "term_id", "term_name", "namespace"],
    columns="alias", values="protein_count", fill_value=0, aggfunc="max",
).reset_index()
for alias in SPECIES:
    if alias not in common_terms:
        common_terms[alias] = 0
shared_terms = common_terms.loc[
    common_terms["mansonia_humeralis"].gt(0) & common_terms["mansonia_titillans"].gt(0)
].sort_values(["category", "mansonia_humeralis", "mansonia_titillans"], ascending=[True, False, False])

top_category_terms = (
    category_term_coverage.groupby(["alias", "category"], group_keys=False)
    .head(8)
    .reset_index(drop=True)
)
display(top_category_terms[[
    "species", "category_label", "term_id", "term_name", "namespace",
    "protein_count", "candidate_count", "fraction_of_category",
]])
display(shared_terms[[
    "category_label", "term_id", "term_name", "namespace",
    "mansonia_humeralis", "mansonia_titillans",
]])

,species,category_label,term_id,term_name,namespace,protein_count,candidate_count,fraction_of_category
0,Mansonia humeralis,Tracheal / spiracle,GO:0043005,neuron projection,cellular_component,4,7,0.571429
1,Mansonia humeralis,Tracheal / spiracle,GO:0031252,cell leading edge,cellular_component,3,7,0.428571
2,Mansonia humeralis,Tracheal / spiracle,GO:0033674,positive regulation of kinase activity,biological_process,3,7,0.428571
3,Mansonia humeralis,Tracheal / spiracle,GO:0006468,protein phosphorylation,biological_process,3,7,0.428571
4,Mansonia humeralis,Tracheal / spiracle,GO:0005524,ATP binding,molecular_function,2,7,0.285714
5,Mansonia humeralis,Tracheal / spiracle,GO:0000981,"DNA-binding transcription factor activity, RNA polymerase II-specific",molecular_function,2,7,0.285714
6,Mansonia humeralis,Tracheal / spiracle,GO:0007169,cell surface receptor protein tyrosine kinase signaling pathway,biological_process,2,7,0.285714
7,Mansonia humeralis,Tracheal / spiracle,GO:0005737,cytoplasm,cellular_component,2,7,0.285714
8,Mansonia humeralis,Cuticle / chitin,GO:0008061,chitin binding,molecular_function,128,175,0.731429
9,Mansonia humeralis,Cuticle / chitin,GO:0062129,chitin-based extracellular matrix,cellular_component,123,175,0.702857


alias,category_label,term_id,term_name,namespace,mansonia_humeralis,mansonia_titillans
40,Tracheal / spiracle,GO:0043005,neuron projection,cellular_component,4,3
17,Tracheal / spiracle,GO:0006468,protein phosphorylation,biological_process,3,5
37,Tracheal / spiracle,GO:0033674,positive regulation of kinase activity,biological_process,3,5
33,Tracheal / spiracle,GO:0031252,cell leading edge,cellular_component,3,2
20,Tracheal / spiracle,GO:0007275,multicellular organism development,biological_process,2,6
...,...,...,...,...,...,...
376,Mechanosensory,GO:0007626,locomotory behavior,biological_process,1,2
359,Mechanosensory,GO:0004984,olfactory receptor activity,molecular_function,1,1
363,Mechanosensory,GO:0005261,monoatomic cation channel activity,molecular_function,1,1
381,Mechanosensory,GO:0008061,chitin binding,molecular_function,1,1


In [10]:
top10_per_protein = (
    top10_specific.groupby(["alias", "protein_id"], as_index=False)
    .agg(most_specific_term_count=("term_id", "nunique"))
)
candidate_category_coverage = (
    candidate_map[["alias", "species", "category", "category_label", "protein_id"]]
    .drop_duplicates(subset=["alias", "protein_id"])
    .merge(top10_per_protein, on=["alias", "protein_id"], how="left", validate="one_to_one")
)
candidate_category_coverage["most_specific_term_count"] = (
    candidate_category_coverage["most_specific_term_count"].fillna(0).astype(int)
)
category_coverage = (
    candidate_category_coverage.groupby(
        ["alias", "species", "category", "category_label"], as_index=False
    ).agg(
        candidate_count=("protein_id", "nunique"),
        candidates_with_any_top10=("most_specific_term_count", lambda values: int(values.gt(0).sum())),
        candidates_with_10_top10=("most_specific_term_count", lambda values: int(values.eq(TOP_SPECIFIC_TERMS).sum())),
        median_top10_terms=("most_specific_term_count", "median"),
    )
)
category_coverage["fraction_with_any_top10"] = (
    category_coverage["candidates_with_any_top10"] / category_coverage["candidate_count"]
)
display(category_coverage.sort_values(["alias", "category"]))

,alias,species,category,category_label,candidate_count,candidates_with_any_top10,candidates_with_10_top10,median_top10_terms,fraction_with_any_top10
0,mansonia_humeralis,Mansonia humeralis,1_tracheal_spiracle,Tracheal / spiracle,7,7,7,10.0,1.0
1,mansonia_humeralis,Mansonia humeralis,2_cuticle_chitin,Cuticle / chitin,175,175,175,10.0,1.0
2,mansonia_humeralis,Mansonia humeralis,3_hypoxia_signaling,Hypoxia signaling,6,6,6,10.0,1.0
3,mansonia_humeralis,Mansonia humeralis,4_chemosensory,Chemosensory,78,78,78,10.0,1.0
4,mansonia_humeralis,Mansonia humeralis,5_mechanosensory,Mechanosensory,17,17,17,10.0,1.0
5,mansonia_titillans,Mansonia titillans,1_tracheal_spiracle,Tracheal / spiracle,9,9,9,10.0,1.0
6,mansonia_titillans,Mansonia titillans,2_cuticle_chitin,Cuticle / chitin,168,168,168,10.0,1.0
7,mansonia_titillans,Mansonia titillans,3_hypoxia_signaling,Hypoxia signaling,6,6,6,10.0,1.0
8,mansonia_titillans,Mansonia titillans,4_chemosensory,Chemosensory,85,85,85,10.0,1.0
9,mansonia_titillans,Mansonia titillans,5_mechanosensory,Mechanosensory,15,15,15,10.0,1.0


In [11]:
matrix_comparison = prediction_qc[[
    "species", "PDF_matrix_rows", "prediction_rows", "row_count_difference"
]].copy()
matrix_comparison["row_count_change_percent"] = (
    100 * matrix_comparison["row_count_difference"] / matrix_comparison["PDF_matrix_rows"]
)
display(matrix_comparison)

candidate_quality = (
    candidate_map.assign(has_internal_stop=candidate_map["internal_stops"].gt(0))
    .groupby(["species", "category_label"], as_index=False)
    .agg(
        candidates=("protein_id", "nunique"),
        candidates_with_internal_stops=("has_internal_stop", "sum"),
        median_reference_identity=("identity", "median"),
    )
)
display(candidate_quality.sort_values(["species", "category_label"]))

alignment_counts = comparison["alignment"].value_counts()
exact_count = int(alignment_counts.get("exact locus and GO term in current top 10", 0))
outside_count = int(alignment_counts.get("term present in matrix, outside current top 10", 0))
absent_count = int(alignment_counts.get("term not found in nonzero prediction matrix", 0))
unresolved_count = int(
    alignment_counts.get("PDF GO name unresolved", 0)
    + alignment_counts.get("PDF locus not in current candidate set", 0)
)
display(Markdown(
    f"### Observed alignment with the PDF\n\n"
    f"Of **{len(comparison)}** term-level calls transcribed from the PDF, "
    f"**{exact_count}** occur for the same locus in the current top 10 most-specific terms; "
    f"**{outside_count}** are present elsewhere in the nonzero prediction matrix; "
    f"**{absent_count}** are not present in that matrix; and **{unresolved_count}** could not be mapped exactly.\n\n"
    f"These counts measure agreement with the report’s selected examples. They are not accuracy metrics."
))

alignment_by_category = (
    comparison.groupby(["category_label", "alignment"]).size().unstack(fill_value=0)
    .reindex(index=list(CATEGORY_LABELS.values()), fill_value=0)
    .reindex(columns=[
        "exact locus and GO term in current top 10",
        "term present in matrix, outside current top 10",
        "term not found in nonzero prediction matrix",
        "PDF GO name unresolved",
        "PDF locus not in current candidate set",
    ], fill_value=0)
    .reset_index()
)
display(alignment_by_category)

,species,PDF_matrix_rows,prediction_rows,row_count_difference,row_count_change_percent
0,Mansonia humeralis,2942351,2964991,22640,0.769453
1,Mansonia titillans,2980556,3003196,22640,0.759590


,species,category_label,candidates,candidates_with_internal_stops,median_reference_identity
0,Mansonia humeralis,Chemosensory,78,7,0.57020
1,Mansonia humeralis,Cuticle / chitin,175,3,0.73330
2,Mansonia humeralis,Hypoxia signaling,6,1,0.64315
3,Mansonia humeralis,Mechanosensory,17,1,0.77370
4,Mansonia humeralis,Tracheal / spiracle,7,0,0.77430
5,Mansonia titillans,Chemosensory,85,4,0.56860
6,Mansonia titillans,Cuticle / chitin,168,1,0.72470
7,Mansonia titillans,Hypoxia signaling,6,0,0.63930
8,Mansonia titillans,Mechanosensory,15,1,0.81440
9,Mansonia titillans,Tracheal / spiracle,9,0,0.76710


### Observed alignment with the PDF

Of **60** term-level calls transcribed from the PDF, **37** occur for the same locus in the current top 10 most-specific terms; **22** are present elsewhere in the nonzero prediction matrix; **0** are not present in that matrix; and **1** could not be mapped exactly.

These counts measure agreement with the report’s selected examples. They are not accuracy metrics.

alignment,category_label,exact locus and GO term in current top 10,"term present in matrix, outside current top 10",term not found in nonzero prediction matrix,PDF GO name unresolved,PDF locus not in current candidate set
0,Tracheal / spiracle,10,2,0,0,0
1,Cuticle / chitin,4,2,0,0,0
2,Hypoxia signaling,14,7,0,1,0
3,Chemosensory,5,6,0,0,0
4,Mechanosensory,4,5,0,0,0


## Full top-10 results for PDF-highlighted loci

This table exposes the current top-10 most-specific terms for every locus named in the PDF, including terms the PDF did not list. It is useful for seeing whether the full-network run changed the profile of a highlighted candidate while keeping the locus fixed.

In [12]:
highlighted_proteins = set(report_claims["protein_id"].dropna())
highlighted_top10 = top10_specific.loc[
    top10_specific["protein_id"].isin(highlighted_proteins)
].copy()
protein_sections = (
    report_claims.dropna(subset=["protein_id"])
    .groupby(["alias", "protein_id"], as_index=False)["section"]
    .first()
)
highlighted_top10 = highlighted_top10.merge(
    protein_sections, on=["alias", "protein_id"], how="left", validate="many_to_one"
).sort_values(["section", "alias", "locus_key", "specific_rank"])
display(highlighted_top10[[
    "section", "species", "locus_key", "category_label", "specific_rank",
    "term_id", "term_name", "namespace", "score", "internal_stops",
]])

,section,species,locus_key,category_label,specific_rank,term_id,term_name,namespace,score,internal_stops
0,4.1 Tracheal / spiracle,Mansonia humeralis,ptg001450l_1:42115-44279(+),Tracheal / spiracle,1,GO:0046983,protein dimerization activity,molecular_function,0.591304,0
1,4.1 Tracheal / spiracle,Mansonia humeralis,ptg001450l_1:42115-44279(+),Tracheal / spiracle,2,GO:0000977,RNA polymerase II transcription regulatory region sequence-specific DNA binding,molecular_function,0.192242,0
2,4.1 Tracheal / spiracle,Mansonia humeralis,ptg001450l_1:42115-44279(+),Tracheal / spiracle,3,GO:0000981,"DNA-binding transcription factor activity, RNA polymerase II-specific",molecular_function,0.192242,0
3,4.1 Tracheal / spiracle,Mansonia humeralis,ptg001450l_1:42115-44279(+),Tracheal / spiracle,4,GO:0005634,nucleus,cellular_component,0.106449,0
4,4.1 Tracheal / spiracle,Mansonia humeralis,ptg001450l_1:42115-44279(+),Tracheal / spiracle,5,GO:0005667,transcription regulator complex,cellular_component,0.106342,0
...,...,...,...,...,...,...,...,...,...,...
325,4.5 Mechanosensory,Mansonia titillans,ptg013409l_1:254107-328314(-),Mechanosensory,6,GO:0051480,regulation of cytosolic calcium ion concentration,biological_process,0.197293,0
326,4.5 Mechanosensory,Mansonia titillans,ptg013409l_1:254107-328314(-),Mechanosensory,7,GO:0098703,calcium ion import across plasma membrane,biological_process,0.100231,0
327,4.5 Mechanosensory,Mansonia titillans,ptg013409l_1:254107-328314(-),Mechanosensory,8,GO:0051262,protein tetramerization,biological_process,0.099905,0
328,4.5 Mechanosensory,Mansonia titillans,ptg013409l_1:254107-328314(-),Mechanosensory,9,GO:0042221,response to chemical,biological_process,0.099799,0


## Interpretation

Observed alignment is partial but clear for the report’s selected examples: 37 of 60 table-level PDF calls recur for the same locus and GO concept in the current top 10; 22 more have a nonzero matrix score outside that top 10; none were absent from the nonzero matrix; and one label could not be mapped exactly. The notebook also breaks those counts down by category. This supports consistency for selected calls, not a general accuracy claim.

The strongest repeated category-level terms in this run include chitin binding among cuticle/chitin candidates (128/175 humeralis; 132/168 titillans), odorant binding among chemosensory candidates (33/78; 34/85), and calcium channel activity among mechanosensory candidates (10/17; 11/15). For the six hypoxia-signaling candidates per species, the most frequent terms differ: response to hypoxia occurs for 5/6 humeralis candidates, while nucleus, response to chemical, and response to temperature stimulus each occur for 5/6 titillans candidates. These are recurrent predictions within preselected categories, not independent evidence that the category labels are biologically correct.

The report’s gene candidates were selected from three culicid reference proteomes before S2F. Full STRING expands the propagation context, but cannot recover loci omitted at that earlier candidate-identification step. The PDF’s raw reduced-network matrices are unavailable here, so this notebook cannot attribute changes to network scope in a controlled comparison. The 22,640 additional prediction rows per species (about 0.76% over the PDF row counts) are a difference in output size, not evidence of better predictions.

Treat S2F scores as relative ranks within a protein. There is no experimental validation set for these Mansonia candidates, and the PDF reports no RNA-seq validation. The extraction summary marks internal-stop gene models; predictions for those proteins deserve extra sequence-level review. These computational hypotheses may help prioritize experiments, but do not replace them.

## Reproduction

Run all cells from the S2F environment with the external disk mounted at the configured path. The notebook reads only existing prediction outputs and source files. It does not regenerate predictions or export derived tables/files outside this notebook.

Recorded prediction commands, interpreter paths, configs, input FASTAs, run times, file sizes, and matrix row counts are shown above. The notebook’s main checks require each manifest and prediction file to exist, each FASTA and extraction summary to contain the expected 283 mapped candidates, the prediction IDs to match those FASTAs exactly, and each PDF term/locus match to be explicitly resolved or marked unresolved.